In [2]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from functools import  partial
import xarray as xr
parent_dir = os.path.dirname(os.environ["GTE_DIR"].replace("Glaciation_time_estimator",""))
GTE_DIR=os.environ["GTE_DIR"]
sys.path.insert(0, parent_dir)
from glaciation_time_estimator.data_postprocessing.Job_result_fp_generator import generate_tracking_filenames
from glaciation_time_estimator.auxiliary_func.config_reader import read_config

In [3]:
config = read_config(
    os.path.join(GTE_DIR,'configs/2021_tracking/01_01.yaml'))
analyze_year=True
year=2022
global global_rmse
global_rmse = config["Global_sqrt_mse"]
ice_cont_crit_frac = 0.05
# classifiacation_palette = ['#e41a1c', '#377eb8', "#4daf4a"]
classifiacation_palette = ['#e41a1c', '#377eb8', "#4daf4a"]


In [4]:
if os.uname()[1]=="n2o":
    combined_cloud_df = pd.read_parquet(f"/net/n2o/wolke_scratch/dnikolo/Final_results/{year}_all.parquet")
if os.uname()[1][:3]=="eu-":
    combined_cloud_df = pd.read_parquet(f"/cluster/work/climate/dnikolo/Cloud_analysis/full_years/{year}_all.parquet")
combined_cloud_df=combined_cloud_df[~combined_cloud_df.is_large_pix_cloud]
combined_cloud_df = combined_cloud_df[(combined_cloud_df.avg_lat >30) | (combined_cloud_df.avg_lat<-30)]

In [5]:
T_tot=365*24
combined_cloud_df["norm_area"] = combined_cloud_df["avg_size[km]"] * combined_cloud_df["Lifetime [h]"]/T_tot
tracked_area = combined_cloud_df['norm_area'].sum()*24/23
print(f"{tracked_area:0e}")

8.153316e+06


9.063577e+06 sq. km. is the average tracked cloud cover in 2022

In [6]:
sp_cover = xr.load_dataset("/wolke_scratch/dnikolo/CLAAS_Data/Cloud_cover/sp/combined_mean.nc")
np_cover = xr.load_dataset("/wolke_scratch/dnikolo/CLAAS_Data/Cloud_cover/np/combined_mean.nc")

In [7]:
mean_cover_2022 = (sp_cover.mean_area.isel(georef_offset_corrected=1)+np_cover.mean_area.isel(georef_offset_corrected=1)).to_numpy()[0,0,0]


In [8]:
sp_total = xr.load_dataset("/wolke_scratch/dnikolo/CLAAS_Data/sp/summed_aux.nc")
np_total = xr.load_dataset("/wolke_scratch/dnikolo/CLAAS_Data/np/summed_aux.nc")
total_field_area = (sp_total.filtered_area.isel(georef_offset_corrected=1)+np_total.filtered_area.isel(georef_offset_corrected=1)).to_numpy()[0,0]

In [9]:
print(f"Cloud cover is {mean_cover_2022/total_field_area*100:02}%")
print(f"We track: {(tracked_area/mean_cover_2022*100):02}\% of the total cloud cover or {tracked_area/total_field_area*100:02}% of total area")

Cloud cover is 31.154412031173706%
We track: 31.74099269902416\% of the total cloud cover or 9.888719618460756% of total area


Cloud cover is 31.154412031173706% of total area\
We track: 30.41845133656482\% of the total cloud cover or 9.476689634358225% of total area